# 07 · SwinUNETR V2 vs U-Net — Deep supervision OFF

**새로 초기화한 SwinUNETR V2 → U-Net**을 GPU 한 장에서 순차 학습합니다. 두 모델 모두 **fold 0 하나·250 epochs·deep supervision OFF·pretrained 없음**입니다. 시작 셀 한 번으로 서버의 tmux 큐가 smoke test → 학습 → final validation → best validation → 다음 모델을 실행합니다. 브라우저를 닫아도 tmux는 유지되지만 서버 인스턴스 종료 시 중단됩니다.

학습은 nnU-Net이 담당합니다. MONAI 1.5.1은 Swin 네트워크 구현만 제공합니다. nnU-Net 기본 전체 증강(감마 포함), Dice+CE, SGD(lr=.01, momentum=.99, Nesterov, wd=3e-5), 250 epoch PolyLR, train 250/val 50 steps, 기존 ACDC spacing·샘플링·분할을 공통 사용합니다. 양쪽 모두 AMP, compile=False. Swin은 공식 기본 feature_size=24, use_v2=True, merging, 그리고 메모리 절약용 gradient checkpointing=True입니다. 이는 논문의 미공개 세부 설정을 정확히 재현했다고 주장하는 실험이 아닙니다.

패치는 `[20,256,224]`로 유지합니다. Swin 내부에서 깊이 앞뒤에 6개씩 0-padding하여 `[32,256,224]`로 만든 뒤 출력의 padding을 잘라 loss/추론에 전달합니다. U-Net에는 추가 padding이 없습니다. 작은 영상의 일반 nnU-Net padding은 양쪽 모두 유지합니다.

대조군도 기존 결과에서 이어 학습하지 않습니다. 기존 1000 epoch 또는 증강 테마 결과는 DS ON이므로 별도의 참고 결과입니다. 새 두 모델의 동일 fold 비교가 주 비교입니다. seed를 고정하지 않는 nnU-Net 기본 동작을 유지하므로 단일 실행 결과에는 무작위 변동이 섞입니다.

공식 근거: [nnU-Net 확장 문서](https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/extending_nnunet.md), [MONAI 1.5.1 네트워크](https://github.com/Project-MONAI/MONAI/blob/1.5.1/monai/networks/nets/swin_unetr.py).

## 0. 환경과 경로

Python 3.11 venv가 이미 있는 서버에서 그 커널을 선택하세요. 기존 Jupyter 서버를 계속 사용할 수 있습니다. `git`, `tmux`, NVIDIA 드라이버가 필요합니다. 이 노트북은 OS 패키지나 드라이버를 자동 설치하지 않습니다.

기존 **전처리 완료된 acdc-data 폴더 전체**를 repo 옆에 복사하면 재다운로드·재전처리 없이 실행합니다. 처음부터 준비하는 경우 01_prepare.ipynb를 먼저 완료하세요. raw validation 영상·정답, 전처리 배열, metadata/acdc/download_manifest.json, plans·split이 필요합니다. 아래 경로를 확인하세요.

새 서버의 빈 venv는 INSTALL_PACKAGES=True로 설치 셀을 한 번 실행한 뒤 커널을 재시작하세요. 다른 실험이 학습 중인 venv에는 패키지를 설치하지 마세요. 이번 실험은 별도 venv를 권장합니다.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shlex, shutil
sys.dont_write_bytecode = True
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'nnunetv2').is_dir()), None)
if REPO is None:
    raise RuntimeError('nnUNet repo 안에서 노트북을 여세요.')
EXPERIMENT = REPO / 'experiments/acdc'
STORAGE_ROOT = (REPO.parent / 'acdc-data').resolve()
SUITE_ID = 'acdc-swinv2-unet-nods250-v1'
GPU = '0'
BATCH_SIZE = 4  # 메모리 부족 시 새 SUITE_ID로 2를 선택. 반드시 두 모델 공통 변경.
AUGMENTATION_WORKERS = 4
EVALUATE_BEST = True  # final과 best를 별도 보관하고 둘 다 비교
INSTALL_PACKAGES = False
START_SUITE = False
RESUME = False
print('Kernel:', sys.executable)
print('Storage:', STORAGE_ROOT)
print('Suite:', SUITE_ID, '| GPU:', GPU, '| common batch:', BATCH_SIZE)

In [ ]:
if INSTALL_PACKAGES:
    required_files = [
        EXPERIMENT / 'requirements-server.txt',
        EXPERIMENT / 'architecture_comparison/requirements.txt',
        EXPERIMENT / 'architecture_comparison/runner.py',
        EXPERIMENT / 'analysis/compare_best.py',
        REPO / 'nnunetv2/training/nnUNetTrainer/variants/network_architecture/nnUNetTrainerACDCNoDS250.py',
    ]
    missing = [str(p) for p in required_files if not p.is_file()]
    if missing:
        raise FileNotFoundError('서버 repo에 필수 파일이 없습니다. 최신 repo 전체를 동기화하세요:\n' + '\n'.join(missing))
    if sys.platform != 'linux' or sys.prefix == sys.base_prefix:
        raise RuntimeError('Linux GPU 서버의 venv에서만 설치하세요.')
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError('이 서버 recipe는 Python 3.11 venv를 기준으로 합니다.')
    if any(name in sys.modules for name in ('torch', 'monai', 'numpy', 'nnunetv2')):
        raise RuntimeError('커널을 재시작하고 초기화 및 설치 셀부터 실행하세요.')
    from importlib.metadata import version, PackageNotFoundError
    specs = ['torch==2.8.0+cu128', 'torchvision==0.23.0+cu128']
    try:
        version('torchaudio')
    except PackageNotFoundError:
        pass
    else:
        specs.append('torchaudio==2.8.0+cu128')
    pip = [sys.executable, '-m', 'pip']
    subprocess.check_call(pip + ['install', '--no-cache-dir', *specs,
                                '--index-url', 'https://download.pytorch.org/whl/cu128'])
    subprocess.check_call(pip + ['install', '--no-cache-dir', *specs, '-e', str(REPO),
                                '-r', str(EXPERIMENT / 'requirements-server.txt'),
                                '-r', str(EXPERIMENT / 'architecture_comparison/requirements.txt')])
    subprocess.check_call(pip + ['check'])
    subprocess.check_call([sys.executable, '-c',
        "import torch, monai; print(torch.__version__, monai.__version__); "
        "assert torch.cuda.is_available(), 'CUDA unavailable: check driver and PyTorch wheel'; "
        "print(torch.cuda.get_device_name(0))"], env=dict(os.environ, CUDA_VISIBLE_DEVICES=GPU))
    print('설치 완료. 커널을 재시작하고 INSTALL_PACKAGES=False로 바꿔 처음부터 실행하세요.')
else:
    print('설치 생략')

## 1. 실험 구성

설정은 실행할 때 저장합니다. 기존 suite와 설정·패키지·코드가 달라지면 재개를 거부합니다. `BATCH_SIZE`, 모델 설정 또는 패키지를 변경했다면 새 `SUITE_ID`를 사용하세요. 새 plans 파일은 기존 전처리 데이터를 참조하며 원래 nnUNetPlans.json은 수정하지 않습니다.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('acdc_architecture_runner', EXPERIMENT / 'architecture_comparison/runner.py')
architecture = importlib.util.module_from_spec(spec)
spec.loader.exec_module(architecture)
from workflow import environment, parse_losses, model_dir, write_json
suite = dict(suite_id=SUITE_ID, storage_root=str(STORAGE_ROOT), gpu=GPU,
             batch_size=BATCH_SIZE, augmentation_workers=AUGMENTATION_WORKERS,
             models=['swin_v2', 'unet'], fold=0, epochs=250,
             evaluate_best=EVALUATE_BEST, execution_enabled=True,
             expected_git_commit=subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
architecture.validate_suite(suite)
SUITE_DIR = architecture.suite_dir(suite)
CONFIG_PATH = SUITE_DIR / 'suite.server.json'
SESSION = 'acdc-arch-' + SUITE_ID
for name in suite['models']:
    print(name, '→', model_dir(architecture.model_config(suite, name)))
print('학습은 아직 시작하지 않았습니다.')

## 2. 한 번에 백그라운드 실행 / 재개

위에서 `START_SUITE=True`로 바꾸고 초기화·구성 셀을 갱신한 뒤 실행하세요. 시작 후 아래 상태 셀에서 runner가 정상인지 확인하세요. 자동 smoke test는 **실제 패치·batch로 가상 데이터 1 step**만 실행하며, 여기서 사용한 가중치는 버립니다. smoke 성공은 전체 학습의 메모리 안정성을 보장하지는 않습니다.

CUDA OOM이면 runner.log를 확인하고 `BATCH_SIZE=2` 및 새 SUITE_ID로 **두 모델을 함께** 다시 구성하세요. 자동으로 batch를 바꿔 서로 다른 조건을 만들지는 않습니다. 중단한 동일 설정을 이어가려면 RESUME=True를 사용합니다. 완료한 모델의 학습은 건너뛰며 best 평가 중단 시 이미 생성된 유효한 예측을 재사용합니다.

In [ ]:
if not START_SUITE:
    print('실행 생략. GPU 서버에서 START_SUITE=True로 변경하세요.')
else:
    if sys.platform != 'linux' or sys.prefix == sys.base_prefix:
        raise RuntimeError('Linux GPU 서버의 venv에서 실행하세요.')
    search_path = str(Path(sys.executable).parent) + os.pathsep + os.environ.get('PATH', '')
    for tool in ('git', 'tmux', 'nvidia-smi', 'nnUNetv2_train'):
        if shutil.which(tool, path=search_path) is None:
            raise RuntimeError(f'서버에 먼저 설치하세요: {tool}')
    if subprocess.run(['tmux', 'has-session', '-t', SESSION], capture_output=True).returncode == 0:
        print('이미 실행 중:', SESSION)
    else:
        SUITE_DIR.mkdir(parents=True, exist_ok=True)
        if CONFIG_PATH.exists() and json.loads(CONFIG_PATH.read_text()) != suite:
            raise ValueError('기존 suite 설정과 다릅니다. 새 SUITE_ID를 사용하세요.')
        write_json(CONFIG_PATH, suite)
        command = [sys.executable, '-u', str(EXPERIMENT / 'architecture_comparison/runner.py'),
                   '--config', str(CONFIG_PATH)]
        if RESUME:
            command.append('--resume')
        runner_log = SUITE_DIR / 'runner.log'
        shell_command = shlex.join(command) + ' >> ' + shlex.quote(str(runner_log)) + ' 2>&1'
        subprocess.run(['tmux', 'new-session', '-d', '-s', SESSION, shell_command],
                       env=environment(architecture.model_config(suite, 'swin_v2')), check=True)
        print('Started:', SESSION)
        print('Log:', runner_log)
        print('아래 상태/로그 셀에서 preflight·smoke 결과를 확인하세요.')

## 3. 상태·메모리 검사·최근 로그

이 셀은 모니터링만 하며 학습을 새로 시작하지 않습니다. 실패한 경우 runner.log와 해당 모델의 fold_0.console.log에 원인이 기록됩니다.

In [ ]:
from collections import deque
status_file = SUITE_DIR / 'status.json'
status = json.loads(status_file.read_text()) if status_file.exists() else {}
print('Status:', status)
if sys.platform == 'linux' and shutil.which('tmux'):
    print('tmux alive:', subprocess.run(['tmux', 'has-session', '-t', SESSION], capture_output=True).returncode == 0)
smoke_file = SUITE_DIR / 'smoke.json'
if smoke_file.exists():
    print('GPU smoke results:', json.loads(smoke_file.read_text()))
for file in [SUITE_DIR / 'runner.log', *[
    STORAGE_ROOT / 'runs' / architecture.model_config(suite, name)['run_id'] / 'logs/fold_0.console.log'
    for name in suite['models']]]:
    if file.exists():
        print('\n', file)
        with file.open(errors='replace') as stream:
            print(''.join(deque(stream, maxlen=14)))

## 4. Train loss와 validation loss

두 loss는 epoch마다 샘플링된 패치의 지표이며, 전체 3D 영상 Dice/mIoU는 다음 셀에서 비교합니다. epoch는 1부터 표시합니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, name in zip(axes, suite['models']):
    history = pd.DataFrame(parse_losses(model_dir(architecture.model_config(suite, name)) / 'fold_0'))
    if not history.empty:
        ax.plot(history['epoch'] + 1, history['train_loss'], label='Train loss')
        ax.plot(history['epoch'] + 1, history['val_loss'], label='Validation loss')
        ax.legend()
        display(history.tail(3))
    ax.set(title=name + ' — no deep supervision', xlabel='Epoch', ylabel='Dice + CE loss')
    ax.grid(alpha=.2)
fig.tight_layout()
plt.show()

## 5. 동일 validation 40개에서 final / best 비교

큐가 자동으로 comparison.csv/json을 갱신합니다. **final끼리, best끼리** 비교하세요. best는 전체 볼륨 Dice가 아니라 학습 중 EMA pseudo-Dice로 선택한 체크포인트입니다. 같은 validation으로 선택과 평가를 하므로 독립 test 결과는 아닙니다. NSD는 이 비교 표에 포함하지 않습니다.

best는 `architecture_suites/<SUITE_ID>/best/<모델>/`에 예측 마스크·사례별 지표를 저장하며 기존 final validation 폴더를 덮어쓰지 않습니다. 확률 배열은 저장하지 않습니다.

In [ ]:
comparison = pd.DataFrame(architecture.comparison_rows(suite))
display(comparison.round(3))
completed = comparison[comparison.status == 'completed']
if not completed.empty:
    for checkpoint in completed.checkpoint.unique():
        table = completed[completed.checkpoint == checkpoint].set_index('model')
        ax = table[['dice_percent', 'miou_percent']].plot.bar(
            figsize=(7, 4), ylim=(0, 100), rot=0, ylabel='Whole-volume score (%)',
            title=f'{checkpoint}: fold 0, 250 epochs, no deep supervision')
        for container in ax.containers:
            ax.bar_label(container, fmt='%.2f', padding=3)
        ax.figure.tight_layout()
        if SUITE_DIR.exists():
            ax.figure.savefig(SUITE_DIR / f'comparison-{checkpoint}.png', dpi=160)
        plt.show()
print('요약:', SUITE_DIR / 'comparison.csv')

## 6. 필요할 때만 중단

STOP_SUITE=True는 실행 중인 큐와 현재 작업에 Ctrl+C를 보냅니다. 마지막 저장 이후 진행분은 유실될 수 있습니다. tmux 종료와 GPU 사용량을 확인한 뒤 RESUME=True로 재개하세요. 중단 시점의 즉시 체크포인트 저장을 보장하지 않습니다.

In [ ]:
STOP_SUITE = False
if STOP_SUITE:
    if sys.platform != 'linux' or not shutil.which('tmux'):
        raise RuntimeError('서버 tmux 환경에서 실행하세요.')
    subprocess.run(['tmux', 'send-keys', '-t', SESSION, 'C-c'], check=True)
    print('중단 요청 전송. 로그와 GPU 상태를 확인하세요.')